# 創薬仮説生成システム
**Drug Discovery Hypothesis Generator**

遺伝子名と疾患名を入力すると、複数の公開DBから根拠データを収集し、LLMが創薬仮説を生成します。

### データソース（全て商用利用可能）
| ソース | 内容 | ライセンス |
|--------|------|------------|
| PubMed (NCBI) | 論文・アブストラクト | パブリックドメイン |
| OpenTargets | 遺伝子-疾患エビデンス統合スコア | Apache 2.0 |
| UniProt | タンパク質機能・構造・局在 | CC BY 4.0 |
| IntAct (EMBL-EBI) | タンパク質相互作用 (PPI) | CC BY 4.0 |
| SIGNOR | 因果シグナル伝達ネットワーク | CC BY 4.0 |
| GWAS Catalog | ゲノムワイド関連解析 | EMBL-EBI (free) |
| ClinVar (NCBI) | 病的変異 | パブリックドメイン |
| ChEMBL | 既存薬・臨床候補・作用機序 | CC BY-SA 3.0 |
| DGIdb | 薬剤-遺伝子相互作用（リポジショニング） | Apache 2.0 |
| gnomAD | 集団変異頻度・制約スコア (pLI/LOEUF) | CC BY 4.0 |
| GTEx | 組織別遺伝子発現プロファイル | CC BY 4.0 |
| Human Protein Atlas | タンパク質発現・病理・細胞内局在 | CC BY-SA 4.0 |
| AlphaFold DB | 3D構造予測・ドラッガビリティ評価 | CC BY 4.0 |
| Reactome | 因果関係付きパスウェイ | CC BY 4.0 |
| ClinicalTrials.gov | 進行中・完了済み臨床試験 | パブリックドメイン |
| PubChem BioAssay (NCBI) | 毒性アッセイデータ | パブリックドメイン |
| openFDA | 副作用自発報告 | パブリックドメイン |
| g:Profiler | GO/Reactome/WikiPathways エンリッチメント | BSD 2-Clause |

### LLM
- **Ollama** (ローカル実行、完全無料) — デフォルト

## Step 1: セットアップ

初回のみ実行してください。

In [ ]:
import subprocess
subprocess.run([
    "pip", "install",
    "requests", "ipywidgets",
    "networkx", "plotly",
], check=True)
print("インストール完了")

## Step 2: Ollama のセットアップ（初回のみ）

### 1. Ollama をインストール
ターミナルで実行：
```bash
curl -fsSL https://ollama.com/install.sh | sh
```
または [ollama.com](https://ollama.com) からインストーラをダウンロード（Mac/Win対応）

### 2. モデルをダウンロード（初回のみ・数分かかります）
ターミナルで実行：
```bash
ollama pull llama3.1
```
- `llama3.1`（8B）: 約4.7GB、推奨
- `llama3.1:70b`: 約40GB、高品質だが大容量

### 3. Ollama サーバーを起動
```bash
ollama serve
```
※ Mac アプリ版をインストールした場合はメニューバーから自動起動されます

In [28]:
# Ollama の起動確認
import requests

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print("✓ Ollama 起動中")
    print(f"  利用可能モデル: {models if models else '（モデル未ダウンロード）'}")
    if not models:
        print("\n⚠ ターミナルで以下を実行してモデルをダウンロードしてください:")
        print("  ollama pull llama3.1")
except Exception:
    print("⚠ Ollama が起動していません")
    print("\n以下をターミナルで実行してください:")
    print("  1. ollama serve      # サーバー起動")
    print("  2. ollama pull llama3.1  # モデルDL（初回のみ）")

✓ Ollama 起動中
  利用可能モデル: ['llama3.1:latest']


## Step 3: LLMクライアントの初期化

In [29]:
import sys
sys.path.insert(0, '.')

for mod in list(sys.modules.keys()):
    if "llm" in mod:
        del sys.modules[mod]

from llm.ollama_client import OllamaClient

# 使用するモデルを指定（ollama pull で取得済みのもの）
MODEL = "llama3.1"   # 他の例: "llama3.1:70b", "mistral", "gemma3"

llm = OllamaClient(model=MODEL)

if llm.is_available():
    print(f"✓ Ollama ({MODEL}) クライアント初期化完了")
else:
    raise RuntimeError(
        "⚠ Ollama に接続できません。\n"
        "ターミナルで 'ollama serve' を実行してから再度試してください。"
    )

✓ Ollama (llama3.1) クライアント初期化完了


## Step 4: 入力設定

遺伝子名を入力し、疾患名は下のウィジェットで検索・選択してください。

In [30]:
import sys, requests
import ipywidgets as widgets
from IPython.display import display
sys.path.insert(0, '.')

OT_API = "https://api.platform.opentargets.org/api/v4/graphql"

def _ot_search(keyword, entity):
    q = """
    query ($q: String!, $e: [String!]) {
      search(queryString: $q, entityNames: $e, page: {index: 0, size: 15}) {
        hits { id name description entity }
      }
    }
    """
    r = requests.post(OT_API, json={"query": q, "variables": {"q": keyword, "e": [entity]}}, timeout=15)
    r.raise_for_status()
    return [h for h in r.json()["data"]["search"]["hits"] if h["entity"] == entity]

# 選択結果を保持
selected_gene    = {"id": None, "name": None}
selected_disease = {"id": None, "name": None}

def make_search_ui(label, entity, selected_dict):
    box  = widgets.Text(placeholder=f"{label}名を入力...",
                        layout=widgets.Layout(width="380px"))
    btn  = widgets.Button(description="検索", button_style="primary",
                          layout=widgets.Layout(width="70px"))
    lst  = widgets.Select(options=[], rows=6,
                          layout=widgets.Layout(width="640px"))
    stat = widgets.Label(value=f"{label}名を入力して「検索」を押してください")

    def on_search(_):
        kw = box.value.strip()
        if not kw:
            stat.value = "⚠ キーワードを入力してください"; return
        stat.value = f"検索中..."
        try:
            hits = _ot_search(kw, entity)
            if not hits:
                stat.value = f"「{kw}」に一致する{label}が見つかりませんでした"
                lst.options = []; return
            lst.options = [
                (f"{h['name']}  [{h['id']}]  {(h.get('description') or '')[:55]}", h)
                for h in hits
            ]
            stat.value = f"{len(hits)} 件見つかりました。リストから選択してください"
        except Exception as e:
            stat.value = f"エラー: {e}"

    def on_select(change):
        val = change["new"]
        if val:
            selected_dict["id"]   = val["id"]
            selected_dict["name"] = val["name"]
            stat.value = f"✓ 選択済み: {val['name']}  ({val['id']})"

    btn.on_click(on_search)
    lst.observe(on_select, names="value")
    return widgets.VBox([widgets.HBox([box, btn]), lst, stat])

display(widgets.VBox([
    widgets.HTML("<b>① 遺伝子</b>（例: BRAF, TP53, EGFR）"),
    make_search_ui("遺伝子", "target", selected_gene),
    widgets.HTML("<hr style='margin:8px 0'><b>② 疾患 / 症状</b>（例: melanoma, Parkinson disease）"),
    make_search_ui("疾患", "disease", selected_disease),
]))

## Step 5: データ収集（全DBへの並列問い合わせ）

In [41]:
import sys

# collectors モジュールのキャッシュをクリアして最新ファイルを強制ロード
for mod in list(sys.modules.keys()):
    if "collectors" in mod or "aggregator" in mod:
        del sys.modules[mod]

from aggregator import collect_all, build_llm_context
import json

# Step 4 のウィジェット変数の存在チェック
try:
    _gene_name    = selected_gene["name"]
    _disease_name = selected_disease["name"]
except NameError:
    raise RuntimeError(
        "⚠ Step 4 のセルが実行されていません。\n"
        "上の「Step 4: 入力設定」セルを実行し、遺伝子と疾患を選択してから再度実行してください。"
    )

GENE    = _gene_name
DISEASE = _disease_name

if not GENE or not DISEASE:
    raise ValueError("⚠ Step 4 で遺伝子と疾患を両方選択してから実行してください")

print(f"解析対象:")
print(f"  遺伝子:  {GENE}  ({selected_gene['id']})")
print(f"  疾患:    {DISEASE}  ({selected_disease['id']})")
print(f"\nデータ収集を開始します...\n")

raw_evidence = collect_all(
    GENE, DISEASE,
    verbose=True,
    gene_id=selected_gene["id"],
    disease_id=selected_disease["id"],
)

print("\nデータ収集完了")
if raw_evidence.get("collection_errors"):
    print("\n⚠ 一部エラー:", raw_evidence["collection_errors"])

解析対象:
  遺伝子:  MARK2  (ENSG00000072518)
  疾患:    Duchenne muscular dystrophy  (MONDO_0010679)

データ収集を開始します...

  [+] gwas: OK
  [+] uniprot: OK
  [+] opentargets: OK
  [+] clinvar: OK
  [+] intact: OK
  [+] chembl: OK
    遺伝子シノニム (2件): MARK2, EMK1
  [+] pubmed: OK
  [+] toxicity: OK

データ収集完了


## Step 5b: PPI ネットワーク構築 (IntAct + SIGNOR + BioGRID)

タンパク質相互作用データから遺伝子のネットワークを構築します。

- **IntAct** (EMBL-EBI) — 実験的PPI、CC BY 4.0
- **SIGNOR** (UNIROMA2) — 因果シグナル伝達ネットワーク、CC BY 4.0
- **BioGRID** — APIキー必要（無料登録: https://webservice.thebiogrid.org/ ）、非商用利用

> BioGRID は非商用・学術用途限定です。APIキーなしの場合は自動でスキップされます。

In [ ]:
import sys
for mod in list(sys.modules.keys()):
    if "network" in mod or "collectors" in mod:
        del sys.modules[mod]

import network as net_mod

# BioGRID APIキーを設定（持っている場合）
# BIOGRID_API_KEY = "your_32char_key_here"
BIOGRID_API_KEY = "e8f7cf92fae447b3e9d6729aa6a815eb"  # None の場合は BioGRID をスキップ

print(f"PPIネットワーク構築中: {GENE}")
print("-" * 40)

ppi_graph = net_mod.build_ppi_network(
    GENE,
    use_biogrid=True,
    biogrid_api_key=BIOGRID_API_KEY,
)

if ppi_graph is not None:
    print(f"\n✓ ネットワーク完成:")
    print(f"  ノード数: {ppi_graph.number_of_nodes()}")
    print(f"  エッジ数: {ppi_graph.number_of_edges()}")
    import networkx as nx
    center = GENE.upper()
    partners = list(ppi_graph.neighbors(center))
    print(f"  直接インタラクター ({len(partners)}件): {', '.join(partners[:15])}{'...' if len(partners)>15 else ''}")
else:
    print("⚠ networkx が未インストール: pip install networkx")

PPIネットワーク構築中: MARK2
----------------------------------------
  IntAct 取得中...
  IntAct: 20 件
  SIGNOR 取得中...
  SIGNOR: 26 件
  BioGRID 取得中...


## Step 5c: エンリッチメント解析 (g:Profiler)

PPIネットワークの全遺伝子を使って GO / KEGG / Reactome / WikiPathways エンリッチメント解析を実行します。

In [ ]:
print(f"エンリッチメント解析中...")
print("-" * 40)

if ppi_graph is not None:
    network_enrichment = net_mod.run_network_enrichment(ppi_graph, top_n=30)
    
    by_src = network_enrichment.get("by_source", {})
    print(f"\n✓ エンリッチメント完成:")
    for src, terms in by_src.items():
        print(f"\n  [{src}] 上位{len(terms)}件:")
        for t in terms[:3]:
            print(f"    {t['term_name'][:60]} (p={t['p_value']:.2e}, n={t['intersection_size']})")
else:
    network_enrichment = {}
    print("⚠ PPIネットワークが未構築のためスキップしました")

エンリッチメント解析中...
----------------------------------------
  エンリッチメント対象: 6 遺伝子
  エンリッチメント: 30 有意項目 (FDR<0.05)

✓ エンリッチメント完成:

  [GO:CC] 上位3件:
    HCN channel complex (p=1.52e-13, n=4)
    voltage-gated potassium channel complex (p=2.85e-10, n=5)
    potassium channel complex (p=3.44e-10, n=5)

  [REAC] 上位1件:
    HCN channels (p=2.29e-12, n=4)

  [GO:BP] 上位5件:
    regulation of membrane depolarization (p=7.40e-11, n=5)
    potassium ion import across plasma membrane (p=7.40e-11, n=5)
    regulation of heart rate by cardiac conduction (p=7.40e-11, n=5)

  [GO:MF] 上位5件:
    voltage-gated potassium channel activity (p=7.77e-10, n=5)
    voltage-gated sodium channel activity (p=7.77e-10, n=4)
    cAMP binding (p=7.79e-10, n=4)


## Step 5d: LLMコンテキスト生成

収集データ + PPIネットワーク + エンリッチメント結果をまとめて LLM 用のコンテキストを生成します。

In [ ]:
from aggregator import build_llm_context

# 基本コンテキスト（DB収集データから）
context = build_llm_context(raw_evidence)

# PPIネットワーク + エンリッチメントサマリーを追記
if ppi_graph is not None:
    network_section = net_mod.network_summary_for_llm(
        ppi_graph, GENE, network_enrichment,
        max_partners=10, max_terms=15,
    )
    context = context + "\n\n" + network_section

print(f"✓ コンテキスト生成完了 ({len(context):,} 文字)")
print("\n--- プレビュー (先頭500字) ---")
print(context[:500])

✓ コンテキスト生成完了 (8,063 文字)

--- プレビュー (先頭500字) ---
# Evidence Summary for Drug Target Hypothesis
Gene: HCN4
Disease/Condition: Duchenne muscular dystrophy
(Each evidence item has a [Ref N] tag — cite these in your report)

## Gene/Protein Information [Ref 1]
- Protein: Potassium/sodium hyperpolarization-activated cyclic nucleotide-gated channel 4
- Function: Hyperpolarization-activated ion channel that are permeable to Na(+) and K(+) ions with very slow activation and inactivation (PubMed:10228147, PubMed:10430953, PubMed:20829353). Exhibits hig


## Step 6: 収集データの確認（オプション）

In [ ]:
ev = raw_evidence["evidence"]

# OpenTargets スコア
ot = ev.get("opentargets") or {}
score = ot.get("association_score")
score_str = f"{score:.3f}" if score is not None else "N/A"
print(f"=== OpenTargets 関連スコア ===")
print(f"  {GENE} × {ot.get('disease_label', DISEASE)}: {score_str}")
for k, v in (ot.get('datatype_scores') or {}).items():
    print(f"    {k}: {v:.3f}")

# 既存薬
drugs = ev.get("chembl") or []
ot_drugs = ot.get("known_drugs") or []
all_drugs = drugs + ot_drugs
print(f"\n=== 既存薬・臨床候補 ({len(all_drugs)}件) ===")
for d in all_drugs[:5]:
    name = d.get('name') or d.get('drug', '')
    phase = d.get('max_phase') or d.get('phase')
    mech = d.get('mechanism') or d.get('mechanism_of_action', '')
    print(f"  {name} | Phase {phase} | {mech}")

# GWAS
gwas_hits = ev.get("gwas") or []
print(f"\n=== GWAS ヒット ({len(gwas_hits)}件) ===")
for h in gwas_hits[:3]:
    print(f"  {h['trait']} | p={h['p_value']}")

# ClinVar
cv = ev.get("clinvar") or []
print(f"\n=== ClinVar 病的変異 ({len(cv)}件) ===")
for v in cv[:3]:
    print(f"  {v['title'][:80]}... | {v['clinical_significance']}")

# 論文
papers = ev.get("pubmed") or []
print(f"\n=== 関連論文 (上位{len(papers)}件) ===")
for p in papers[:3]:
    print(f"  [{p['year']}] {p['title'][:80]}...")

=== OpenTargets 関連スコア ===
  HCN4 × Duchenne muscular dystrophy: N/A

=== 既存薬・臨床候補 (7件) ===
  IVABRADINE HYDROCHLORIDE | Phase 4.0 | Potassium/sodium hyperpolarization-activated cyclic nucleotide-gated channel 4 blocker
  DRONEDARONE HYDROCHLORIDE | Phase 4.0 | Potassium/sodium hyperpolarization-activated cyclic nucleotide-gated channel 4 blocker
  IVABRADINE | Phase 4.0 | Potassium/sodium hyperpolarization-activated cyclic nucleotide-gated channel 4 blocker
  IVABRADINE HYDROCHLORIDE | Phase APPROVAL | 
  IVABRADINE | Phase APPROVAL | 

=== GWAS ヒット (0件) ===

=== ClinVar 病的変異 (10件) ===
  NM_005477.3(HCN4):c.1591-1861_2867del... | 
  NM_005477.3(HCN4):c.712A>C (p.Lys238Gln)... | 
  NM_005477.3(HCN4):c.146G>T (p.Arg49Leu)... | 

=== 関連論文 (上位1件) ===
  [2022] Involvement of muscle satellite cell dysfunction in neuromuscular disorders: Exp...


## Step 6b: 関連論文の表示

In [ ]:
from IPython.display import HTML, display

papers = raw_evidence["evidence"].get("pubmed") or []

RATING_COLOR = {4: "#1a7f37", 3: "#0969da", 2: "#9a6700", 1: "#b35900", 0: "#888"}
RATING_LABEL = {
    4: "★★★★ 公式シンボル×タイトル",
    3: "★★★☆ 公式シンボル×アブスト",
    2: "★★☆☆ シノニム×タイトル",
    1: "★☆☆☆ シノニム×アブスト",
    0: "☆☆☆☆ 内容参照",
}

cards = []
for p in papers:
    score  = p.get("relevance_score", 0)
    color  = RATING_COLOR[score]
    label  = RATING_LABEL[score]
    match  = p.get("match_type", "")
    abstract_short = (p.get("abstract") or "アブストラクトなし")[:300]
    if len(p.get("abstract", "")) > 300:
        abstract_short += "..."
    authors = ", ".join(p.get("authors", []))
    if len(p.get("authors", [])) >= 3:
        authors += " et al."

    cards.append(f"""
<div style="border:1px solid #ddd;border-left:5px solid {color};border-radius:6px;
            padding:12px 16px;margin:8px 0;background:#fafafa">
  <div style="display:flex;justify-content:space-between;align-items:center">
    <span style="font-size:11px;font-weight:bold;color:{color}">{label}</span>
    <span style="font-size:11px;color:#666">{p.get('year','')} | {p.get('journal','')[:50]}</span>
  </div>
  <div style="font-weight:bold;margin:6px 0 4px;font-size:14px">{p.get('title','')}</div>
  <div style="font-size:12px;color:#555;margin-bottom:6px">{authors}</div>
  <div style="font-size:12px;color:#333;line-height:1.5">{abstract_short}</div>
  <div style="margin-top:6px">
    <a href="https://pubmed.ncbi.nlm.nih.gov/{p['pmid']}/" target="_blank"
       style="font-size:11px;color:#0969da">PubMed: {p['pmid']} →</a>
  </div>
</div>""")

html = f"""
<h3 style="margin-bottom:4px">関連論文 ({len(papers)}件) — {GENE} × {DISEASE}</h3>
<p style="font-size:12px;color:#666;margin-top:0">
  関連度順（新しい順）。★4=公式シンボルがタイトルに一致、★3=アブストに一致、★2-1=シノニム一致、★0=内容参照
</p>
{"".join(cards) if cards else "<p>論文が見つかりませんでした</p>"}
"""
display(HTML(html))

## Step 6c: PPIネットワーク可視化

> **注意**: pyvis が必要です。未インストールの場合: `pip install pyvis networkx`

In [ ]:
import subprocess
subprocess.run(["pip", "install", "plotly", "networkx", "--quiet"], check=False)

# 表示するインタラクター数（多すぎると重くなります。推奨: 30〜60）
MAX_NODES = 50

if ppi_graph is not None and ppi_graph.number_of_nodes() > 0:
    fig = net_mod.visualize_network_plotly(
        ppi_graph, GENE,
        enrichment=network_enrichment,
        max_nodes=MAX_NODES,
    )
    if fig is not None:
        fig.show()
    else:
        print("⚠ plotly 未インストール。`pip install plotly` で追加できます。")
else:
    print("⚠ PPIネットワークが空です。Step 5b を先に実行してください。")

## Step 6d: エンリッチメント解析結果の表示

In [ ]:
from IPython.display import HTML, display

SOURCE_COLOR = {
    "GO:BP": "#4ECDC4", "GO:MF": "#45B7D1", "GO:CC": "#96CEB4",
    "REAC":  "#FF6B6B", "WP":    "#DDA0DD",
}
SOURCE_LABEL = {
    "GO:BP": "GO Biological Process",
    "GO:MF": "GO Molecular Function",
    "GO:CC": "GO Cellular Component",
    "REAC":  "Reactome",
    "WP":    "WikiPathways",
}

enrich_results = (network_enrichment or {}).get("results", [])

if not enrich_results:
    display(HTML("<p>エンリッチメント結果がありません。Step 5c を実行してください。</p>"))
else:
    by_src = (network_enrichment or {}).get("by_source", {})
    sections = []

    for src, terms in by_src.items():
        color = SOURCE_COLOR.get(src, "#CCC")
        label = SOURCE_LABEL.get(src, src)
        rows = []
        for t in terms:
            pv  = t["p_value"]
            pct = int(t["gene_ratio"] * 100)
            genes_str = ", ".join(t.get("genes", [])[:8])
            if len(t.get("genes", [])) > 8:
                genes_str += "..."
            rows.append(f"""
            <tr>
              <td style="padding:6px 10px;font-weight:bold">{t['term_name'][:65]}</td>
              <td style="padding:6px 10px;text-align:center;font-family:monospace">{pv:.2e}</td>
              <td style="padding:6px 10px;text-align:center">{t['intersection_size']}</td>
              <td style="padding:6px 10px">
                <div style="background:#eee;border-radius:4px;height:12px;width:100%">
                  <div style="background:{color};height:12px;border-radius:4px;width:{min(pct*2,100)}%"></div>
                </div>
                <span style="font-size:10px">{pct}%</span>
              </td>
              <td style="padding:6px 10px;font-size:11px;color:#555">{genes_str}</td>
            </tr>""")

        sections.append(f"""
        <div style="margin:12px 0">
          <div style="background:{color};color:#fff;padding:6px 12px;border-radius:6px 6px 0 0;
                      font-weight:bold;font-size:13px">{label}</div>
          <table style="width:100%;border-collapse:collapse;border:1px solid #ddd;border-top:none">
            <thead>
              <tr style="background:#f5f5f5;font-size:12px">
                <th style="padding:6px 10px;text-align:left">Term</th>
                <th style="padding:6px 10px">p-value (FDR)</th>
                <th style="padding:6px 10px">Genes</th>
                <th style="padding:6px 10px">Ratio</th>
                <th style="padding:6px 10px;text-align:left">Intersecting genes</th>
              </tr>
            </thead>
            <tbody>{"".join(rows)}</tbody>
          </table>
        </div>""")

    html = f"""
    <h3>エンリッチメント解析結果 — {GENE} PPIネットワーク ({len(network_enrichment.get('gene_list',[]))} 遺伝子)</h3>
    <p style="font-size:12px;color:#666">g:Profiler / FDR補正済み (Benjamini-Hochberg) / p&lt;0.05</p>
    {"".join(sections)}
    """
    display(HTML(html))

Term,p-value (FDR),Genes,Ratio,Intersecting genes
HCN channel complex,1.52e-13,4,66%,
voltage-gated potassium channel complex,2.85e-10,5,83%,
potassium channel complex,3.44e-10,5,83%,
Term,p-value (FDR),Genes,Ratio,Intersecting genes
HCN channels,2.29e-12,4,66%,
Term,p-value (FDR),Genes,Ratio,Intersecting genes
regulation of membrane depolarization,7.40e-11,5,83%,
potassium ion import across plasma membrane,7.40e-11,5,83%,
regulation of heart rate by cardiac conduction,7.40e-11,5,83%,
cardiac muscle cell action potential,1.05e-09,5,83%,


In [ ]:
import sys
for mod in list(sys.modules.keys()):
    if "hypothesis" in mod:
        del sys.modules[mod]

from hypothesis import generate_hypothesis, generate_presentation_eval
from IPython.display import Markdown, display, clear_output
import ipywidgets as widgets

# ── 言語設定 ──────────────────────────────────────────────
LANG = "ja"   # "ja" = 日本語 / "en" = English

# ── ストリーミング出力 ──────────────────────────────────────
out_stream = widgets.Output()
display(out_stream)

buffer = []

def stream_callback(token: str):
    buffer.append(token)
    with out_stream:
        print(token, end="", flush=True)

# ── 仮説生成 ──────────────────────────────────────────────
lang_label = "日本語" if LANG == "ja" else "English"
print(f"仮説生成中 ({lang_label})... [{type(llm).__name__}]")
print("生成されたテキストがリアルタイムで表示されます...\n")

hypothesis = generate_hypothesis(
    GENE, DISEASE, context, llm,
    lang=LANG,
    stream_callback=stream_callback,
)

# ── ストリーム終了後にMarkdownで再描画 ──────────────────────
out_stream.clear_output()
print(f"\n✓ 生成完了 ({len(hypothesis):,} 文字)")
display(Markdown(hypothesis))

## Step 7: 仮説生成

`LANG = "ja"` で日本語、`"en"` で英語レポートを生成します。

- **Ollama (ローカル)**: トークンがリアルタイムで表示されます（ストリーミング）
- 生成時間の目安: llama3.1 8B ≈ 2〜4分

> 💡 **高速化**: `ollama pull gemma3:4b` や `phi4-mini` など軽量モデルを試してください

In [ ]:
import json
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 遺伝子名・疾患名ごとのサブフォルダに保存
pair_dir = Path("reports") / f"{GENE}_{DISEASE.replace(' ', '_')}"
pair_dir.mkdir(parents=True, exist_ok=True)

# ---- 仮説レポート ----
lang_suffix = "JA" if LANG == "ja" else "EN"
report_path = pair_dir / f"{timestamp}_{lang_suffix}.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"# 創薬仮説レポート: {GENE} × {DISEASE}\n")
    f.write(f"生成日時: {datetime.now().isoformat()}  |  言語: {LANG}\n\n---\n\n")
    f.write(hypothesis)
    f.write("\n\n---\n\n## エビデンスコンテキスト\n\n")
    f.write(context)
print(f"✓ レポート保存: {report_path}")

# ---- 評価カード (JSON) — Step 7b 未実行の場合はスキップ ----
try:
    eval_path = pair_dir / f"{timestamp}_eval.json"
    with open(eval_path, "w", encoding="utf-8") as f:
        json.dump(eval_result, f, ensure_ascii=False, indent=2)
    print(f"✓ 評価データ: {eval_path}")
except NameError:
    print("⚠ 評価データ未生成 (Step 7b を実行すると保存されます)")

# ---- 生データ (JSON) ----
raw_path = pair_dir / f"{timestamp}_raw.json"
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(raw_evidence, f, ensure_ascii=False, indent=2, default=str)
print(f"✓ 生データ: {raw_path}")

print(f"\n📁 フォルダ: {pair_dir.resolve()}")

In [ ]:
# Step 7b: プレゼンテーション用 エビデンス評価カード
# (Step 7 の hypothesis 変数を使用)
from hypothesis import generate_presentation_eval
import json
from IPython.display import HTML, display

print(f"評価カード生成中... [{type(llm).__name__}]")

eval_result = generate_presentation_eval(GENE, DISEASE, context, llm, lang=LANG)

if not eval_result:
    print("⚠ 評価カード生成に失敗しました（JSON解析エラーの可能性）")
else:
    RATING_ICON = {"✅ 強い": "✅", "✅ Strong": "✅",
                   "🟡 中程度": "🟡", "🟡 Moderate": "🟡",
                   "🔴 弱い": "🔴", "🔴 Weak": "🔴",
                   "⬜ データなし": "⬜", "⬜ No data": "⬜"}
    RATING_BG   = {"✅": "#e6ffed", "🟡": "#fff8c5", "🔴": "#ffebe9", "⬜": "#f6f8fa"}

    LABELS = {
        "genetic_association":    "遺伝的関連",
        "functional_association": "機能的関連",
        "clinical_relevance":     "臨床的関連",
        "network_context":        "ネットワーク",
        "target_validity_overall":"ターゲット妥当性",
        "repositioning_potential":"リポジショニング",
        "safety_risk":            "安全性リスク",
        "modality_fit":           "モダリティ適合",
        "overall_confidence":     "総合信頼度",
    }

    cards_html = []
    for key, label in LABELS.items():
        item = eval_result.get(key, {})
        rating = item.get("rating", "⬜ データなし")
        finding = item.get("finding", "-")
        icon = RATING_ICON.get(rating, "⬜")
        bg   = RATING_BG.get(icon, "#f6f8fa")
        cards_html.append(f"""
        <div style="border:1px solid #ddd;border-radius:8px;padding:10px 14px;
                    background:{bg};min-width:200px;flex:1">
          <div style="font-size:11px;font-weight:bold;color:#555;margin-bottom:4px">{label}</div>
          <div style="font-size:22px;margin-bottom:4px">{icon}</div>
          <div style="font-size:12px;color:#333;line-height:1.4">{finding}</div>
        </div>""")

    display(HTML(f"""
    <h3 style="margin-bottom:8px">エビデンス評価サマリー — {GENE} × {DISEASE}</h3>
    <div style="display:flex;flex-wrap:wrap;gap:8px">{"".join(cards_html)}</div>
    """))

    # 変数に保存（Step 8 の保存セルで使用）
    eval_en = eval_result
    eval_ja = eval_result

## Step 8: レポートの保存

## 補足: 複数遺伝子・疾患の一括処理

In [ ]:
# 複数ペアを処理したい場合は以下を参考にしてください

# targets = [
#     ("BRAF", "melanoma"),
#     ("EGFR", "lung adenocarcinoma"),
#     ("GBA", "Parkinson disease"),
# ]

# for gene, disease in targets:
#     print(f"\n{'='*60}")
#     print(f"処理中: {gene} × {disease}")
#     raw = collect_all(gene, disease, verbose=False)
#     ctx = build_llm_context(raw)
#     hyp = generate_hypothesis(gene, disease, ctx, llm)
#     # 保存処理...
#     print(f"完了: {gene} × {disease}")

print("上記のコードのコメントを外して実行してください")

上記のコードのコメントを外して実行してください
